# Lab #6: Keras MLP for Regression

**Objective:** Implement a Multi-Layer Perceptron (MLP) using Keras/TensorFlow for a regression problem and analyze the effect of different activation functions and loss functions on model performance.

**Dataset:** California Housing dataset

In [ ]:
# If using Google Colab, TensorFlow is usually already installed.
# Uncomment if required:
# !pip install tensorflow scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("TensorFlow Version:", tf.__version__)

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("Dataset Shape:", df.shape)
display(df.head())

In [ ]:
print("Features:")
print(housing.feature_names)

print("\nTarget Variable:")
print(housing.target_names)

print("\nTarget Description:")
print("Median house value in units of $100,000")

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df["MedHouseVal"], bins=50)
plt.title("Distribution of House Prices")
plt.xlabel("Median House Value ($100,000)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Scaled training shape:", X_train_scaled.shape)

## MLP Model Development

The model contains an input layer, three hidden layers, and one output neuron for regression.

In [ ]:
def build_model(activation="relu", loss="mse"):
    model = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],)),
        layers.Dense(64, activation=activation),
        layers.Dense(32, activation=activation),
        layers.Dense(16, activation=activation),
        layers.Dense(1)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=loss,
        metrics=["mae"]
    )
    return model

model = build_model("relu", "mse")
model.summary()

## Experiment 1: Activation Functions

Compare ReLU, Sigmoid, and Tanh while keeping the other important hyperparameters consistent.

In [ ]:
activation_functions = ["relu", "sigmoid", "tanh"]
activation_results = {}
activation_histories = {}

for activation in activation_functions:
    print("=" * 60)
    print("Training:", activation)

    model = build_model(activation=activation, loss="mse")

    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=32,
        verbose=0
    )

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    activation_results[activation] = {
        "MAE": mae, "RMSE": rmse, "R2": r2
    }
    activation_histories[activation] = history

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}")

activation_df = pd.DataFrame(activation_results).T
display(activation_df)

In [ ]:
plt.figure(figsize=(12, 7))
for activation, history in activation_histories.items():
    plt.plot(history.history["val_loss"], label=activation)

plt.title("Validation Loss for Different Activation Functions")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## Experiment 2: Loss Functions

Compare MSE, MAE, and Huber Loss.

In [ ]:
loss_functions = {
    "MSE": "mse",
    "MAE": "mae",
    "Huber": keras.losses.Huber()
}

loss_results = {}
loss_histories = {}

for loss_name, loss_function in loss_functions.items():
    print("=" * 60)
    print("Training:", loss_name)

    model = build_model(activation="relu", loss=loss_function)

    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=32,
        verbose=0
    )

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    loss_results[loss_name] = {
        "MAE": mae, "RMSE": rmse, "R2": r2
    }
    loss_histories[loss_name] = history

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}")

loss_df = pd.DataFrame(loss_results).T
display(loss_df)

In [ ]:
plt.figure(figsize=(12, 7))
for loss_name, history in loss_histories.items():
    plt.plot(history.history["val_loss"], label=loss_name)

plt.title("Validation Loss for Different Loss Functions")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## Complete Activation + Loss Comparison

In [ ]:
experiments = [
    ("ReLU", "MSE", "relu", "mse"),
    ("Sigmoid", "MSE", "sigmoid", "mse"),
    ("Tanh", "MSE", "tanh", "mse"),
    ("ReLU", "MAE", "relu", "mae"),
    ("ReLU", "Huber", "relu", keras.losses.Huber()),
    ("Sigmoid", "MAE", "sigmoid", "mae"),
    ("Sigmoid", "Huber", "sigmoid", keras.losses.Huber()),
    ("Tanh", "MAE", "tanh", "mae"),
    ("Tanh", "Huber", "tanh", keras.losses.Huber())
]

all_results = []
all_histories = {}

for activation_name, loss_name, activation, loss in experiments:
    print(f"Training: {activation_name} + {loss_name}")

    model = build_model(activation=activation, loss=loss)

    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=32,
        verbose=0
    )

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    all_results.append({
        "Activation": activation_name,
        "Loss Function": loss_name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

    all_histories[f"{activation_name} + {loss_name}"] = history

comparison_df = pd.DataFrame(all_results).sort_values(
    by="RMSE", ascending=True
).reset_index(drop=True)

display(comparison_df)

In [ ]:
best_model_result = comparison_df.iloc[0]

print("BEST MODEL")
print("=" * 50)
print("Activation Function:", best_model_result["Activation"])
print("Loss Function:", best_model_result["Loss Function"])
print("MAE:", round(best_model_result["MAE"], 4))
print("RMSE:", round(best_model_result["RMSE"], 4))
print("R²:", round(best_model_result["R²"], 4))

In [ ]:
best_experiment_name = (
    best_model_result["Activation"] + " + " +
    best_model_result["Loss Function"]
)
best_history = all_histories[best_experiment_name]

plt.figure(figsize=(12, 7))
plt.plot(best_history.history["loss"], label="Training Loss")
plt.plot(best_history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss - " + best_experiment_name)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## Final Model Evaluation

In [ ]:
best_activation = best_model_result["Activation"].lower()
best_loss_name = best_model_result["Loss Function"]

if best_loss_name == "MSE":
    best_loss = "mse"
elif best_loss_name == "MAE":
    best_loss = "mae"
else:
    best_loss = keras.losses.Huber()

final_model = build_model(
    activation=best_activation,
    loss=best_loss
)

final_history = final_model.fit(
    X_train_scaled, y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=32,
    verbose=1
)

y_pred_final = final_model.predict(
    X_test_scaled, verbose=0
).flatten()

final_mse = mean_squared_error(y_test, y_pred_final)
final_rmse = np.sqrt(final_mse)
final_mae = mean_absolute_error(y_test, y_pred_final)
final_r2 = r2_score(y_test, y_pred_final)

print("\nFINAL MODEL PERFORMANCE")
print("=" * 50)
print("MSE :", round(final_mse, 4))
print("RMSE:", round(final_rmse, 4))
print("MAE :", round(final_mae, 4))
print("R²  :", round(final_r2, 4))

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(y_test, y_pred_final, alpha=0.5)

plt.xlabel("Actual House Value")
plt.ylabel("Predicted House Value")
plt.title("Actual vs Predicted House Values")

min_value = min(y_test.min(), y_pred_final.min())
max_value = max(y_test.max(), y_pred_final.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.grid(True)
plt.show()

In [ ]:
errors = y_test - y_pred_final

plt.figure(figsize=(10, 6))
plt.hist(errors, bins=50)

plt.title("Prediction Error Distribution")
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
prediction_df = pd.DataFrame({
    "Actual Value": y_test.values,
    "Predicted Value": y_pred_final,
    "Error": y_test.values - y_pred_final
})

display(prediction_df.head(20))

## Conclusion

A Multi-Layer Perceptron regression model was successfully implemented using Keras/TensorFlow on the California Housing dataset. Different activation functions (ReLU, Sigmoid, and Tanh) and regression loss functions (MSE, MAE, and Huber) were evaluated.

The models were compared using MAE, RMSE, and R². The model with the best overall regression performance was selected based on low error values and a high R² score.

The learning curves were analyzed to identify convergence, overfitting, and underfitting behavior. Huber Loss can provide greater robustness to outliers than MSE because it transitions from squared-error behavior to absolute-error behavior for larger errors.

In [ ]:
# Save the trained final model
final_model.save("keras_mlp_regression_model.keras")
print("Final model saved as keras_mlp_regression_model.keras")